In [1]:
import torch
from datasets import load_dataset,Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType


In [2]:
import pandas as pd
import numpy as np
import json
import os

dataset_path = "dataset/all_combined.csv"
df_dataset = pd.read_csv(dataset_path)

In [3]:
target = "Atheism"           # e.g., "Atheism" or a key from TARGETS_MAP
dataset = "semeval"          # e.g., "semeval" or "wtwt"
type_split = "train" 
filtered_df = df_dataset[
    (df_dataset["target"] == target) &
    (df_dataset["dataset"] == dataset) &
    (df_dataset["type"] == type_split)
]

In [4]:
stance_labels = ["FAVOR", "AGAINST", "NONE"]  # Adjust as needed
def make_prompt(row):
    return f"Tweet: {row['text']}\nWhat is the stance? Options: {', '.join(stance_labels)}"

finetune_df = pd.DataFrame({
    "prompt": filtered_df.apply(make_prompt, axis=1),
    "completion": filtered_df["stance"]
})


In [5]:


# Choose your model
model_name = "meta-llama/Llama-3.2-3B-Instruct"  # or "mistralai/Mistral-7B-Instruct-v0.3"

# Load dataset
dataset = Dataset.from_pandas(finetune_df)



In [6]:

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_8bit=True,  # Optional: saves memory
    device_map="auto"
)



The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
# LoRA config
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # works for Llama/Mistral, check your model if unsure
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)



In [8]:
# Tokenization
def preprocess(example):
    prompt = example["prompt"]
    completion = example["completion"]
    # Concatenate prompt and answer, with a separator if you like
    full_text = prompt + "\n" + completion
    return tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_dataset = dataset.map(preprocess, batched=False)



Map:   0%|          | 0/513 [00:00<?, ? examples/s]

In [9]:
# Training arguments
output_dir = 'lora_model'
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    output_dir=output_dir,
    save_total_limit=2,
    logging_steps=10,
    save_steps=100,
    report_to="none"
)



In [ ]:
# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Train
trainer.train()

# Save final model (LoRA adapters)
trainer.save_model(output_dir)
print(f"LoRA fine-tuned model saved to {output_dir}")

/tmp/ipykernel_3744485/3035406824.py:8: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
10,3.914600
20,3.055700
30,2.421800
40,2.088400
50,2.092200
